# 04 — Optimization & Simulation

Score and rank work orders, schedule technicians, and simulate station operations to evaluate the scheduling strategy before live deployment.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from optimization.priority import WorkOrder, rank_orders
from optimization.scheduler import Technician, Station, assign_orders
from optimization.simulator import StationSimulator

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Sample work orders
orders = [
    WorkOrder('WO001', 'STN_001', battery_health_pct=30, inventory_level_pct=10,
              sla_remaining_hours=2, station_traffic_rank=0.9, predicted_demand_4h=20),
    WorkOrder('WO002', 'STN_002', battery_health_pct=70, inventory_level_pct=60,
              sla_remaining_hours=12, station_traffic_rank=0.3, predicted_demand_4h=5),
    WorkOrder('WO003', 'STN_003', battery_health_pct=50, inventory_level_pct=25,
              sla_remaining_hours=4, station_traffic_rank=0.6, predicted_demand_4h=12),
]
ranked = rank_orders(orders)
ranked

In [ ]:
# Schedule technicians
techs = [
    Technician('TECH_A', current_lat=30.05, current_lon=31.24,
               shift_end=datetime.now() + timedelta(hours=8)),
    Technician('TECH_B', current_lat=30.02, current_lon=31.28,
               shift_end=datetime.now() + timedelta(hours=8)),
]
stations = {
    'STN_001': Station('STN_001', lat=30.06, lon=31.22),
    'STN_002': Station('STN_002', lat=30.01, lon=31.30),
    'STN_003': Station('STN_003', lat=30.04, lon=31.25),
}
schedule = assign_orders(ranked, techs, stations)
schedule

In [ ]:
# Simulate station operations
sim = StationSimulator(horizon_hours=24, seed=42)
for sid, cap in [('STN_001', 20), ('STN_002', 15), ('STN_003', 18)]:
    sim.add_station(sid, capacity=cap, initial_inventory=cap)

demand_rates = {'STN_001': 6.0, 'STN_002': 3.0, 'STN_003': 4.5}
results = sim.run(demand_rates, restock_hours=4.0)
results

In [ ]:
# Fulfillment rate bar chart
results.set_index('station_id')['fulfillment_rate'].plot(
    kind='bar', ylim=(0, 1), title='Station Fulfillment Rate (24h sim)', ylabel='Rate'
)
plt.tight_layout()